In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("Libraries imported successfully!")

Libraries imported successfully!


In [2]:
PROJECT_ROOT = Path(r"D:\Pune-House-Price-Prediction")

MODEL_DIR = PROJECT_ROOT / "models"
MODEL_DIR.mkdir(parents=True, exist_ok=True)

TUNED_MODEL_PATH = MODEL_DIR / "tuned_model_candidate.joblib"
TUNING_REPORT_PATH = MODEL_DIR / "tuning_report.json"
FEATURE_CONFIG_PATH = MODEL_DIR / "model_features.json"

FINAL_MODEL_PATH = MODEL_DIR / "best_house_price_model.joblib"
FINAL_REPORT_PATH = MODEL_DIR / "final_model_report.json"

for path in [TUNED_MODEL_PATH, TUNING_REPORT_PATH, FEATURE_CONFIG_PATH]:
    if not path.exists():
        raise FileNotFoundError(f"Required file not found: {path}")

print("All required files found.")

All required files found.


In [3]:
final_model = joblib.load(TUNED_MODEL_PATH)

with open(TUNING_REPORT_PATH, "r", encoding="utf-8") as file:
    tuning_report = json.load(file)

with open(FEATURE_CONFIG_PATH, "r", encoding="utf-8") as file:
    feature_config = json.load(file)

FEATURES = feature_config["features"]
AMENITY_COLUMNS = feature_config["amenity_columns"]

print("Selected model:", tuning_report["selected_model"])
print("Number of features:", len(FEATURES))
print("Number of amenities:", len(AMENITY_COLUMNS))

Selected model: Extra Trees
Number of features: 50
Number of amenities: 35


In [4]:
model_feature_names = list(final_model.feature_names_in_)

if model_feature_names != FEATURES:
    raise ValueError(
        "Feature mismatch! Model input features and configuration do not match."
    )

print("Feature configuration verified.")

Feature configuration verified.


In [5]:
joblib.dump(final_model, FINAL_MODEL_PATH)

print("Final model saved successfully!")
print("Path:", FINAL_MODEL_PATH)
print("File size (MB):", round(FINAL_MODEL_PATH.stat().st_size / (1024 * 1024), 2))

Final model saved successfully!
Path: D:\Pune-House-Price-Prediction\models\best_house_price_model.joblib
File size (MB): 65.25


In [6]:
final_report = {
    "project": "Pune House Price Prediction",
    "model_type": tuning_report["selected_model"],
    "model_file": FINAL_MODEL_PATH.name,
    "dataset_file": tuning_report["dataset_file"],
    "training_rows": tuning_report["training_rows"],
    "testing_rows": tuning_report["testing_rows"],
    "number_of_features": len(FEATURES),
    "number_of_amenities": len(AMENITY_COLUMNS),
    "best_parameters": tuning_report["best_parameters"],
    "cross_validation_mae": tuning_report["best_cross_validation_mae"],
    "test_mae": tuning_report["test_mae"],
    "test_rmse": tuning_report["test_rmse"],
    "test_r2": tuning_report["test_r2"],
    "currency": "INR"
}

with open(FINAL_REPORT_PATH, "w", encoding="utf-8") as file:
    json.dump(final_report, file, indent=4)

print(json.dumps(final_report, indent=4))

{
    "project": "Pune House Price Prediction",
    "model_type": "Extra Trees",
    "model_file": "best_house_price_model.joblib",
    "dataset_file": "pune_cleaned_reference.csv",
    "training_rows": 22160,
    "testing_rows": 5540,
    "number_of_features": 50,
    "number_of_amenities": 35,
    "best_parameters": {
        "model__n_estimators": 100,
        "model__min_samples_split": 5,
        "model__min_samples_leaf": 1,
        "model__max_features": 1.0,
        "model__max_depth": 25
    },
    "cross_validation_mae": 1936425.0161758407,
    "test_mae": 1915397.2874487618,
    "test_rmse": 4777894.059308242,
    "test_r2": 0.9305595697331278,
    "currency": "INR"
}


In [7]:
loaded_model = joblib.load(FINAL_MODEL_PATH)

print("Model loaded from disk successfully.")
print("Model type:", type(loaded_model).__name__)

Model loaded from disk successfully.
Model type: Pipeline


In [8]:
DATA_CANDIDATES = [
    PROJECT_ROOT / "data" / "processed" / "pune_cleaned_reference_v2.csv",
    PROJECT_ROOT / "data" / "processed" / "pune_cleaned_reference.csv"
]

DATA_PATH = next(
    (path for path in DATA_CANDIDATES if path.exists()),
    None
)

if DATA_PATH is None:
    raise FileNotFoundError("Cleaned dataset not found.")

df = pd.read_csv(DATA_PATH)

TARGET = feature_config["target"]
df[TARGET] = pd.to_numeric(df[TARGET], errors="coerce")

valid_df = df.loc[df[TARGET].notna() & (df[TARGET] > 0)].copy()

X_test_sample = valid_df[FEATURES].iloc[[0]].copy()
actual_price = float(valid_df[TARGET].iloc[0])

predicted_price = float(loaded_model.predict(X_test_sample)[0])

print(f"Actual price: ₹{actual_price:,.0f}")
print(f"Predicted price: ₹{predicted_price:,.0f}")
print(f"Absolute error: ₹{abs(actual_price - predicted_price):,.0f}")

Actual price: ₹27,000,000
Predicted price: ₹27,000,000
Absolute error: ₹0


In [9]:
required_files = [
    FINAL_MODEL_PATH,
    FINAL_REPORT_PATH,
    FEATURE_CONFIG_PATH,
    MODEL_DIR / "tuning_report.json"
]

for path in required_files:
    print(f"{'OK' if path.exists() else 'MISSING'}: {path}")

OK: D:\Pune-House-Price-Prediction\models\best_house_price_model.joblib
OK: D:\Pune-House-Price-Prediction\models\final_model_report.json
OK: D:\Pune-House-Price-Prediction\models\model_features.json
OK: D:\Pune-House-Price-Prediction\models\tuning_report.json
